* Baseline 1 – TF-IDF + Cosine
- Có pipeline rõ ràng
- Chạy trên dataset thật
- Sinh top-K job
- Có output để so sánh

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
jobs = pd.read_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/it_jobs.xlsx")
resumes = pd.read_csv("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

job_texts = (
    jobs["title"].fillna("") + " " + jobs["cleaned_description"].fillna("")
).apply(clean_text)


In [3]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000
)

job_vectors = vectorizer.fit_transform(job_texts)


In [4]:
def recommend_jobs_from_cv(cv_index, top_k=10):
    cv_text = clean_text(resumes.loc[cv_index, "Resume"])
    cv_vector = vectorizer.transform([cv_text])
    scores = cosine_similarity(cv_vector, job_vectors).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = jobs.loc[top_idx, ["title", "company", "location"]].copy()
    results["score"] = scores[top_idx]
    return results


In [ ]:
import os

for cv_index in [0, 5, 20, 100]:
    results = recommend_jobs_from_cv(cv_index=cv_index, top_k=10)
    output_path = f"/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_tfidf/jobrec_tfidf_cv{cv_index}_top10.csv"
    results.to_csv(output_path, index=False)
    print("Saved:", output_path)

# tạo folder outputs nếu chưa có
os.makedirs("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_tfidf", exist_ok=True)

# lưu file
output_path = f"/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/candidate/jobrec_tfidf/jobrec_tfidf_cv{cv_index}_top{top_k}.csv"
results.to_csv(output_path, index=False)

print("Saved to:", output_path)

results


Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/jobrec_tfidf_cv0_top10.csv
Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/jobrec_tfidf_cv5_top10.csv
Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/jobrec_tfidf_cv20_top10.csv
Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/jobrec_tfidf_cv100_top10.csv
Saved to: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/jobrec_tfidf_cv100_top10.csv


,title,company,location,score
12419,Intake Specialist Needed at Personal Injury La...,M&Y Personal Injury Lawyers,"Los Angeles, CA, USA",0.139900
7969,President and CEO,Monterey College of Law,"Seaside, CA, USA",0.101167
1317,Enterprise Network Engineer,Kennesaw State University,"Kennesaw, GA, US",0.098676
10549,Director of Applications - Law Firm,NicoServ,"Minneapolis, MN, US",0.097353
7847,Senior Attorney,Puget Law Group,"Tacoma, WA, USA",0.096846
12765,Computer Support Technician,Mendocino College,"Ukiah, CA, US",0.096323
4181,IT Systems Specialist,GGRM Law Firm,"Las Vegas, NV, US",0.094682
2424,Computer User Support Specialist (Remote),Dutch Ridge Consulting Group,"Washington, DC, US",0.094667
10168,Audio Systems Engineer,Augusta University,"Augusta, GA, US",0.094283
16400,Student Services Advisor,Westcliff University,"Irvine, CA, USA",0.094087
